In [ ]:

from typing import TypedDict, Literal

from anyio.lowlevel import checkpoint
from dotenv import load_dotenv
from fastmcp.client.transports import config
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.channels import topic
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.types import interrupt, Command
from rich import print as rprint
from sqlalchemy.sql.functions import user

# ========== 1. 基础配置 ==========
load_dotenv(override=True)

# ========== 3. 大模型初始化 ==========
model = ChatOpenAI(
    model="deepseek-v4-flash",
    temperature=0.3,
    extra_body={
        "thinking": {"type": "disabled"}
    }
)

class OverAllState(TypedDict):
    topic: str
    poem: str
    is_approved: bool

def approve_node(state:OverAllState) -> Command[Literal["llm_node", "default_node"]]:
    is_approved = interrupt("是否同意调用模型？")
    goto = "llm_node" if is_approved else "default_node"
    return Command(
        goto=goto,
        update={"is_approved":is_approved}
    )

def llm_node(state: OverAllState) -> OverAllState:
    topic=state["topic"]
    res = model.invoke([HumanMessage(content=f"帮我写一首关于{topic}主题的七言绝句，只写诗句，不需要赏析")]).content

    return {
        "poem": res
    }

def default_node(state: OverAllState) -> OverAllState:
    return {
        "poem": "请求被拒绝"
    }

# ========== 5. 构建工作流图 ==========
builder = StateGraph(state_schema=OverAllState)

# 添加节点
builder.add_node("approve_node", approve_node)
builder.add_node("llm_node", llm_node)
builder.add_node("default_node", default_node)
builder.add_edge(START, "approve_node")
builder.add_edge("llm_node", END)
builder.add_edge("default_node", END)
# 编译图，绑定检查点和长期存储

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

from IPython.display import display
display(graph)

# 执行第一次图，触发中断
config = {
    "configurable": {
        "thread_id": "123"
    }
}

interrupt_res = graph.invoke({"topic": "菊花"}, config=config)

print("完整状态：", interrupt_res)



In [8]:
user_approved =input("是否同意调用模型？（y/n）").strip().lower() == 'y'

approved_res=graph.invoke(Command(resume=user_approved), config=config)

print(approved_res)


{'topic': '菊花', 'poem': '《咏菊》\n西风飒飒满庭栽，冷艳幽香带露开。\n莫道秋光无觅处，寒芳独向霜天来。', 'is_approved': True}


In [11]:
config1 = {
    "configurable": {
        "thread_id": "234"
    }
}

graph.invoke({"topic": "梅花"}, config=config1)
user_approved1 =input("是否同意调用模型？（y/n）").strip().lower() == 'y'
approved_res1=graph.invoke(Command(resume=user_approved1), config=config1)
print(approved_res1)

{'topic': '梅花', 'poem': '请求被拒绝', 'is_approved': False}
